
# Fiducial 밝기 조정 (수동)

노출과 이미지 저장은 직접 하고, 여기서는 저장된 FITS 한 장씩 점검한다. 흐름:
**3절에 이미지 경로를 넣고 실행 → 그림·표를 보고 그룹 밝기를 조정 → 다시 찍고 3절을 다시 실행** (점검한 이미지는 `history` 에 쌓여 4절에서 비교).

3절이 보여 주는 것
- 전체 영상 위에 fiducial 마다 `ID: peak` (그룹별 색, 포화는 빨간 고리), fiducial 별 crop (같은 밝기 눈금)
- 그룹별 peak 막대 그림 (포화 수준·검출 문턱 선, 못 찾은 것은 x)
- 표 `T` (id, group, peak = 창 안 최대 화소, flux = 배경 뺀 합, nsat = 포화 화소 수) 와 그룹 요약 `S`

기준: 측정에 쓸 노출에서 모든 fiducial 이 **포화되지 않으면서** (peak < 65000) 가장 어두운 것도 **threshold 보다 충분히 밝게** (몇 배).
포화된 것끼리는 peak 이 모두 65535 라 flux 로 비교한다. fiber 가 같이 찍힌 이미지면 `TARGET_FILE` 을 준다.


## 촬영 (take_exposure.py)
노출은 `kspec_metrology/notebooks/take_exposure.py` 로 직접 찍는다 (노출시간 [ms]; 이름 끝에 `_0` 이 자동으로 붙는다; `--dry-run` 은 이름만 확인).
```bash
python kspec_metrology/notebooks/take_exposure.py --exptime 2 10 --out '/data/{date}_fid_{ms}ms.fits'   # fiducial 만 켜고
```
→ `..._fid_2ms_0.fits`, `..._fid_10ms_0.fits`. 밝기를 바꾼 뒤 다시 찍고 3절을 다시 실행한다.


In [ ]:

import os, sys, json, glob
PKG_DIR  = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))   # 이 노트북이 kspec_metrology/notebooks/ 에 있을 때 저장소 루트
OUT_DIR  = os.path.expanduser('~/metrology_out')                   # <- 결과를 쓸 폴더

# ---- 설정: 바꾸면 커널을 재시작한다 (표와 제외 목록은 import 때 읽힌다) ----
FIBER_TABLE   = None            # None = 패키지 기본 표 (config73). 다른 표면 경로
# 빈 리스트 [] 는 패키지 기본값(analysis/fiber_config.py, 설치표 2026-10-12 기준) 을 그대로 쓴다. 적으면 그것이 우선.
BROKEN_FIBERS = []              # 빛이 안 나는 positioner 홀 ID. 기본: 분광기에 fiber 가 안 연결된 33 홀, 끊어진 G4 포함 (= cfg.FIBER_EXCLUDE)
FIXED_FIBERS  = []              # 켜져 있지만 움직이지 않는 positioner 홀 ID. 기본: ['G7', 'H8', 'J6'] (sky fiber 가 꽂힌 고정 positioner)
SKY_FIBERS    = []              # 표에서 -9 인 홀의 sky fiber. 기본: ['B7', 'H1', 'N7']
ANGLE_SIGN    = (1, 1)          # 출력 각도 = 부호 x 계산각 (alpha 반시계 양, beta >= 0)

if FIBER_TABLE:
    os.environ['KSPEC_FIBER_TABLE'] = FIBER_TABLE
os.environ['KSPEC_FIBER_EXCLUDE'] = ','.join(BROKEN_FIBERS)
os.environ['KSPEC_FIBER_FIXED']   = ','.join(FIXED_FIBERS)
os.environ['KSPEC_SKY_FIBERS']    = ','.join(SKY_FIBERS)
os.environ['KSPEC_ANGLE_SIGN']    = f'{ANGLE_SIGN[0]},{ANGLE_SIGN[1]}'
os.environ['KSPEC_USE_MEASURED']  = '1'
os.makedirs(OUT_DIR, exist_ok=True)
sys.path.insert(0, PKG_DIR)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 80
from kspec_metrology import lab
from kspec_metrology.analysis import fiber_config as cfg
print(f'fiber table {cfg.FIBER_TABLE_PATH}: {len(cfg.fiber_ids())} positioners')
print('broken', cfg.FIBER_EXCLUDE, '| fixed', cfg.FIBER_FIXED, '| sky', cfg.SKY_FIBERS, '| angle sign', cfg.ANGLE_SIGN)


In [ ]:

MAGNIFICATION  = None           # camera -> focal 배율 초기값. None = 패키지 기본 (2026-01 commissioning fit -8.86, 설치된 실카메라). 10/17 시험 셋업 영상은 8.40
MODEL          = 'full+affine'  # 본 장비. fiducial 30개 기준 leave-one-out 이 가장 좋았다 (README 표). 실험실 광학계는 'radial+affine'
NWINDOW        = 40
NITER_RECENTER = 1
THRESHOLD      = 1e3            # 시작 문턱 [ADU, 배경 포함]; 검출 개수에 맞춰 자동으로 올리고 내린다
MIN_DIST       = 80             # 이보다 가까운 검출은 하나로 합침 [px]
FINDER_OPTS    = {'minarea': 9, 'deblend_cont': 1.0}
N_EXTRA        = 0              # FIXED/SKY 외의 여분 밝은 점 (반사광 등)
SAT_LEVEL      = 65000.
TILE           = 'tile01'



## 1. 그룹
`fiber_config.FIDUCIAL_GROUPS` (4개 그룹, 30개). 다르면 여기서 dict 로 바꾼다.


In [ ]:

FID_GROUPS  = cfg.FIDUCIAL_GROUPS          # {'G1': [...], 'G2': [...], 'G3': [...], 'G4': [...]}
TARGET_FILE = None                         # fiber 도 같이 찍힌 이미지면 object.info 경로
ids, xf, yf = lab.fiducial_positions()
for g, m in FID_GROUPS.items():
    print(f'{g}: {len(m)}  {m}')
missing = set(ids) - {m for v in FID_GROUPS.values() for m in v}
print('not in any group:', sorted(missing) if missing else 'none')
history = {}                               # 점검한 이미지 -> 표 (4절 비교용)



## 2. 검출만 먼저 (선택)
이미지에서 peak 검출만 해 본다 (문턱값 탐색 없이). 개수가 30 근처인지, spot 이 쪼개지는지(주황) 본다. 이상하면 `THRESHOLD`, `MIN_DIST`, `FINDER_OPTS` 를 바꾼다.


In [ ]:

IMAGE = '/path/to/fid_0.fits'              # 점검할 fiducial 노출 (이름이 ...0.fits 로 끝나야 한다)
Tp = lab.check_peaks(IMAGE, threshold=THRESHOLD, min_dist=MIN_DIST, finder_opts=FINDER_OPTS)



## 3. 밝기 점검 (조정할 때마다 다시 실행)


In [ ]:

IMAGE = IMAGE                              # 새로 찍은 이미지면 경로를 바꾼다
T, S = lab.fiducial_brightness(IMAGE, threshold=THRESHOLD, min_dist=MIN_DIST, finder_opts=FINDER_OPTS, nwindow=NWINDOW,
                               magnification=MAGNIFICATION, groups=FID_GROUPS, sat_level=SAT_LEVEL, target_file=TARGET_FILE)
lab.plot_group_brightness(T, FID_GROUPS, sat_level=SAT_LEVEL, threshold=THRESHOLD, title=os.path.basename(IMAGE))
history[os.path.basename(IMAGE)] = T
display(S.round(0))
F_ = T[T.id != '?']
print('saturated:', F_.id[F_.nsat > 0].tolist() or 'none', '| dimmest:', F_.sort_values('peak').id[:3].tolist(), '| brightest:', F_.sort_values('peak').id[-3:].tolist())
F_.round(2)



### 그룹 하나만 자세히
한 그룹의 fiducial crop 과 수치만 본다 (그룹 밝기를 바꾼 뒤 그 그룹만 확인할 때).


In [ ]:

GROUP = 'G1'
Tg, _ = lab.fiducial_brightness(IMAGE, threshold=THRESHOLD, min_dist=MIN_DIST, finder_opts=FINDER_OPTS, nwindow=NWINDOW,
                                magnification=MAGNIFICATION, groups={GROUP: FID_GROUPS[GROUP]}, sat_level=SAT_LEVEL,
                                target_file=TARGET_FILE, show=False)
Tg = Tg[Tg.group == GROUP].sort_values('peak')
lab.plot_group_brightness(Tg, {GROUP: FID_GROUPS[GROUP]}, sat_level=SAT_LEVEL, threshold=THRESHOLD, title=f'{GROUP}: {os.path.basename(IMAGE)}')
Tg.round(2)



## 4. 점검한 이미지들 비교
조정 전후 / 노출별 peak 을 fiducial 마다 나란히 (포화되지 않은 노출에서는 노출 시간에 비례해야 한다). `history` 를 비우려면 `history = {}`.


In [ ]:

cmp = lab.compare_brightness(history, FID_GROUPS, sat_level=SAT_LEVEL, threshold=THRESHOLD)
cmp.round(0)



## 5. 측정 노출 확인
밝기를 정한 뒤, 실제 측정에 쓸 fiducial 노출(1–2장)로 본 패키지의 fiducial 검출·매칭이 30/30 인지 본다 (`Metrology_Debug.ipynb` 와 같은 함수).


In [ ]:

FID_IMAGES = [IMAGE]                       # 측정에 쓸 fiducial 노출 1-2장
for f in FID_IMAGES:
    T, S = lab.fiducial_brightness(f, threshold=THRESHOLD, min_dist=MIN_DIST, finder_opts=FINDER_OPTS, nwindow=NWINDOW,
                                   magnification=MAGNIFICATION, groups=FID_GROUPS, sat_level=SAT_LEVEL, target_file=TARGET_FILE, show=False)
    n_found = int((T.id != '?').sum()); n_sat = int(((T.id != '?') & (T.nsat > 0)).sum())
    print(f'{os.path.basename(f)}: {n_found}/{len(ids)} fiducials, {n_sat} saturated ->', 'OK' if (n_found == len(ids) and n_sat == 0) else 'adjust or use two exposures')
